# Notebook 1: NumPy + einsum

**Goal:** Eliminate Julia-to-Python friction so you're not hesitating on syntax under pressure.

**Key Julia→Python differences:**
- 0-indexed, row-major, `*` is element-wise, `@` is matmul
- Broadcasting aligns from the **right** (trailing axes must match or be 1)
- `axis=0` collapses rows → fewer rows in result (unless `keepdims=True`)

**Before starting:** skim [NumPy broadcasting docs](https://numpy.org/doc/stable/user/basics.broadcasting.html) — 5 minutes is enough.


In [1]:
import sys
print(sys.executable)

/opt/miniconda3/bin/python3.9


In [2]:
import numpy as np

def softmax(z, axis=-1):
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def check(name, got, expected=None, cond=None, rtol=1e-4, atol=1e-6):
    try:
        if cond is not None:
            passed = bool(cond(got))
        elif expected is not None:
            passed = np.allclose(np.array(got, dtype=float),
                                 np.array(expected, dtype=float), rtol=rtol, atol=atol)
        else:
            passed = True
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}")
        if not passed and expected is not None:
            g, e = np.array(got), np.array(expected)
            print(f"         got shape {g.shape}, expected shape {e.shape}")
            if g.shape == e.shape and g.size <= 16:
                print(f"         got:      {g.ravel()}")
                print(f"         expected: {e.ravel()}")
        return passed
    except Exception as exc:
        print(f"  [ERROR] {name}: {type(exc).__name__}: {exc}")
        return False

print("Helper loaded. softmax and check() are available.")


Helper loaded. softmax and check() are available.


## Ex 1: Row-normalize

Normalize each row of a matrix so it sums to 1.
This is softmax without the exp — same shape/axis logic.

**Gotcha:** use `axis=-1`, not `axis=1`. Use `keepdims=True` or the division will broadcast wrong.


In [11]:
def row_normalize(A):
    """
    A: (..., n) any shape, normalize along last axis.
    Must work on 2D and 3D inputs. No loops.
    """
    return A / np.sum(A, axis=-1, keepdims=True)

#     print(A.shape)
    raise NotImplementedError("your code here")


In [12]:
print("Ex 1: row_normalize")
A = np.array([[1., 2., 3.], 
              [4., 1., 1.]])
r = row_normalize(A)
check("2D rows sum to 1", r.sum(axis=-1), np.ones(2))
check("2D values correct", r[0], np.array([1/6, 2/6, 3/6]))
B = np.random.rand(4, 3, 5)
r3 = row_normalize(B)
check("3D rows sum to 1", r3.sum(axis=-1), np.ones((4,3)))
check("3D shape preserved", cond=lambda x: x.shape == B.shape, got=r3)
# GOTCHA: if you forgot keepdims, 3D will silently give wrong shape


Ex 1: row_normalize
  [PASS] 2D rows sum to 1
  [PASS] 2D values correct
  [PASS] 3D rows sum to 1
  [PASS] 3D shape preserved


True

## Ex 2: Pairwise cosine similarity

Given X of shape (n, d), return (n, n) cosine similarity matrix.
`C[i,j] = X[i]·X[j] / (‖X[i]‖ ‖X[j]‖)`

**Hint:** normalize rows first, then use matrix multiply.


In [55]:
def pairwise_cosine(X):
    """X: (n, d). Returns (n, n) cosine similarity matrix."""
    X = X / np.linalg.norm(X, axis=-1, keepdims=True)
#     print((X@X.T).shape)
    return X@X.T
    raise NotImplementedError("your code here")


In [56]:
np.linalg.norm(X, axis=-1, keepdims=True)

array([[1.        ],
       [1.        ],
       [1.41421356]])

In [57]:
print("Ex 2: pairwise_cosine")
X = np.array([[1., 0.], [0., 1.], [1., 1.]])
C = pairwise_cosine(X)
check("shape (3,3)", cond=lambda x: x.shape == (3,3), got=C)
check("diagonal is 1", C.diagonal(), np.ones(3))
check("orthogonal vectors → 0", C[0,1], 0.0, atol=1e-6)
check("45° → 1/√2", C[0,2], 1/np.sqrt(2))
check("symmetric", C, C.T)
# GOTCHA: n=1 edge case
C1 = pairwise_cosine(np.array([[3., 4.]]))
check("n=1 shape (1,1)", cond=lambda x: x.shape == (1,1), got=C1)
check("n=1 value 1.0", C1[0,0], 1.0)


Ex 2: pairwise_cosine
  [PASS] shape (3,3)
  [PASS] diagonal is 1
  [PASS] orthogonal vectors → 0
  [PASS] 45° → 1/√2
  [PASS] symmetric
  [PASS] n=1 shape (1,1)
  [PASS] n=1 value 1.0


True

## Ex 3: einsum reference — run and read

Run this cell. Every pattern below must be second nature.


In [58]:
A = np.random.randn(3, 4); B = np.random.randn(4, 5)
v = np.random.randn(4);    w = np.random.randn(3)
X = np.random.randn(3, 4)

examples = [
    ("matmul A@B",              "ik,kj->ij",   (A, B)),
    ("inner product",           "i,i->",       (v, v)),
    ("outer product",           "i,j->ij",     (w, v)),
    ("trace",                   "ii->",        (np.eye(4),)),
    ("row norms²",              "ij,ij->i",    (X, X)),
    ("bilinear w·A·v",          "i,ij,j->",    (w, A, v)),
]
for name, sub, ops in examples:
    res = np.einsum(sub, *ops)
    shape = np.array(res).shape
    print(f"  {name:30s}  {sub:20s}  → {shape}")


  matmul A@B                      ik,kj->ij             → (3, 5)
  inner product                   i,i->                 → ()
  outer product                   i,j->ij               → (3, 4)
  trace                           ii->                  → ()
  row norms²                      ij,ij->i              → (3,)
  bilinear w·A·v                  i,ij,j->              → ()


## Ex 4: Batched attention scores

Q shape `(B, H, T, d_k)`, K shape `(B, H, T, d_k)`.
Return scaled scores `(B, H, T, T)` with a **single einsum call**.

Score[b,h,i,j] = Q[b,h,i,:]·K[b,h,j,:] / √d_k


In [59]:
def batched_attention_scores(Q, K):
    """Q, K: (B, H, T, d_k). Returns (B, H, T, T) scaled scores."""
    return np.einsum("bhid,bhjd->bhij", Q,K)/np.sqrt(K.shape[-1])

In [60]:
print("Ex 4: batched_attention_scores")
B, H, T, dk = 2, 4, 6, 8
Q = np.random.randn(B, H, T, dk); K = np.random.randn(B, H, T, dk)
S = batched_attention_scores(Q, K)
check("shape (B,H,T,T)", cond=lambda x: x.shape == (B,H,T,T), got=S)
manual = (Q[0,0] @ K[0,0].T) / np.sqrt(dk)
check("matches manual [0,0]", S[0,0], manual)
# GOTCHA: divide by sqrt(dk) not dk; check variance
check("scale correct (var ~1)", cond=lambda x: 0.3 < np.var(x) < 3.0, got=S)


Ex 4: batched_attention_scores
  [PASS] shape (B,H,T,T)
  [PASS] matches manual [0,0]
  [PASS] scale correct (var ~1)


True

## Ex 5: Circuit scores via W_QK

The QK circuit matrix is `W_QK = W_Q @ W_K.T` (d_model × d_model).
Attention score between positions i and j: `X[i] @ W_QK @ X[j]`

Compute the (T, T) score matrix with a **single einsum** (no intermediate `X @ W_QK`).


In [111]:
def circuit_scores(X, W_QK, d_k):
    """
    X: (T, d_model), W_QK: (d_model, d_model), d_k: int for scaling.
    Returns (T, T) — do NOT compute X @ W_QK as a separate step.
    """
    return np.einsum("id,de,ej->ij", X,W_QK,X.T)/np.sqrt(d_k)

In [112]:
X[1].shape

(8,)

In [113]:
# np.einsum("ik,kd,dj->ij")

In [114]:
print("Ex 5: circuit_scores")
T, dm, dk = 5, 8, 4
X = np.random.randn(T, dm)
W_Q = np.random.randn(dm, dk); W_K = np.random.randn(dm, dk)
W_QK = W_Q @ W_K.T
S = circuit_scores(X, W_QK, dk)
check("shape (T,T)", cond=lambda x: x.shape == (T,T), got=S)
direct = (X @ W_Q) @ (X @ W_K).T / np.sqrt(dk)
check("matches direct computation", S, direct)


Ex 5: circuit_scores
  [PASS] shape (T,T)
  [PASS] matches direct computation


True

## Extra einsum drills

Six patterns. Write the einsum string yourself — don't look at the reference sheet.


In [116]:

# Setup
import numpy as np
A = np.random.randn(4, 5)      # (m, n)
b = np.random.randn(5)         # (n,)
A_batch = np.random.randn(3, 4, 5)   # (B, m, n)
b_batch = np.random.randn(3, 5)      # (B, n)
X = np.random.randn(6, 8)     # (n, d)
A_sq = np.random.randn(3, 5, 5)     # (B, n, n)
u = np.random.randn(4)         # (m,)
v = np.random.randn(5)         # (n,)
U = np.random.randn(3, 4)     # (B, m)
V = np.random.randn(3, 5)     # (B, n)
B_mat = np.random.randn(5, 7) # (n, p)
c = np.random.randn(7)        # (p,)
X3 = np.random.randn(3, 6, 8) # (B, T, d)
W = np.random.randn(8, 8)     # (d, d)
print("Setup done.")


Setup done.


In [120]:
?np.allclose

In [118]:

# Ex 6a: matrix-vector product  A (m,n) @ b (n,) → (m,)
# Ex 6b: batched matrix-vector  A_batch (B,m,n) @ b_batch (B,n) → (B,m)

matvec       = np.einsum("mn,n->m",A,b)  # replace with einsum
matvec_batch = np.einsum("bmn,bn->bm",A_batch,b_batch)  # replace with einsum

# Tests
assert np.allclose(matvec, A @ b),                    "6a failed"
assert np.allclose(matvec_batch[1], A_batch[1] @ b_batch[1]), "6b failed"
print("[PASS] 6a and 6b")


[PASS] 6a and 6b


In [123]:

# Ex 7: covariance matrix  X (n,d) → X.T @ X  shape (d,d)

cov = np.einsum("ni,nj->ij",X,X)  # replace with einsum

assert np.allclose(cov, X.T @ X), "7 failed"
print("[PASS] 7")


[PASS] 7


In [124]:

# Ex 8: batched trace  A_sq (B,n,n) → (B,)  one trace per matrix

traces = np.einsum("bii->b", A_sq)  # replace with einsum

assert np.allclose(traces[0], np.trace(A_sq[0])), "8 failed"
assert traces.shape == (3,), "8 shape failed"
print("[PASS] 8")


[PASS] 8


In [126]:

# Ex 9: batch of outer products  U (B,m), V (B,n) → (B,m,n)

outers = np.einsum("bm,bn->bmn", U,V)  # replace with einsum

assert np.allclose(outers[0], np.outer(U[0], V[0])), "9 failed"
assert outers.shape == (3, 4, 5), "9 shape failed"
print("[PASS] 9")


[PASS] 9


In [127]:

# Ex 10: three-tensor chain  A (m,n) @ B_mat (n,p) @ c (p,) → (m,)  single einsum

chain = np.einsum("mn,np,p->m", A, B_mat, c)  # replace with einsum

assert np.allclose(chain, A @ B_mat @ c), "10 failed"
print("[PASS] 10")


[PASS] 10


In [129]:

# Ex 11: batched projection  X3 (B,T,d) projected by W (d,d) → (B,T,d)

projected = np.einsum("bti,id->btd", X3, W)  # replace with einsum

assert np.allclose(projected[0], X3[0] @ W), "11 failed"
assert projected.shape == (3, 6, 8), "11 shape failed"
print("[PASS] 11")


[PASS] 11
